# DeepGuard AI — Phase 2 Fine-tuning
## Teaching the model to detect Face-Swap deepfakes

**Why:** Original model was trained on StyleGAN2 fakes (99.78% acc).  
Face-swap deepfakes (inswapper, FaceSwap) have different artifacts → need mixed training.

**Strategy:**
- Start from the already fine-tuned checkpoint (don't train from scratch)
- Mix data: DeepGuard face-swap dataset + StyleGAN2 subset
- Fine-tune 8 epochs at **low LR (5e-5)** to avoid forgetting StyleGAN2 knowledge
- Backbone frozen for first 2 epochs (head only), then unfreeze all

**Expected outcome:** Model detects BOTH GAN fakes AND face-swap deepfakes

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────
!pip install timm huggingface_hub kagglehub Pillow torchvision -q
print('✅ Packages installed')

In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    print('Drive already mounted')

import os
SAVE_DIR = '/content/drive/MyDrive/deepguard_finetune'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Save directory: {SAVE_DIR}')

In [ ]:
# ── Cell 3: Config ───────────────────────────────────────────────
HF_TOKEN   = 'YOUR_HF_TOKEN_HERE'
HF_REPO    = 'Sowaiba01/deepguard-ai'
DATA_REPO  = 'Sowaiba01/Deepguard-dataset'

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 8
LR         = 5e-5      # Low LR — we're fine-tuning a fine-tuned model
FREEZE_EPOCHS = 2      # Freeze backbone for first 2 epochs
VAL_SPLIT  = 0.15
KAGGLE_STYLEGAN_LIMIT = 4000  # How many StyleGAN2 fakes to mix in

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Config: IMG={IMG_SIZE}, BS={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LR}')

In [ ]:
# ── Cell 4: Download pre-trained model from HuggingFace ──────────
from huggingface_hub import hf_hub_download

print('Downloading pre-trained DeepGuard checkpoint...')
ckpt_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='efficientnet_b4_deepguard.pth',
    token=HF_TOKEN,
    local_dir='/content/models'
)
print(f'✅ Checkpoint: {ckpt_path}')

In [ ]:
# ── Cell 5: Define model (exact same architecture as training) ────
import torch.nn as nn
import timm

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=False,
            num_classes=0,
            global_pool='avg'
        )
        self.head = nn.Sequential(
            nn.Linear(1792, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),  nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

model = DeepfakeDetector()
state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model = model.to(DEVICE)
print(f'✅ Model loaded — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
# ── Cell 6: Download DeepGuard face-swap dataset from HuggingFace ─
from huggingface_hub import snapshot_download
import glob

print('Downloading DeepGuard face-swap dataset (~280MB)...')
dataset_dir = snapshot_download(
    repo_id=DATA_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/content/deepguard_dataset'
)

# Find fake and real folders
fake_paths = sorted(glob.glob(f'{dataset_dir}/fake/*.jpg') + 
                    glob.glob(f'{dataset_dir}/fake/*.png'))
real_paths = sorted(glob.glob(f'{dataset_dir}/real/*.jpg') + 
                    glob.glob(f'{dataset_dir}/real/*.png'))

print(f'✅ Face-swap fakes: {len(fake_paths)}')
print(f'✅ Real images:     {len(real_paths)}')

In [ ]:
# ── Cell 7: Download StyleGAN2 subset from Kaggle (for balance) ───
# This prevents the model from forgetting how to detect GAN fakes
import kagglehub, random

print(f'Downloading StyleGAN2 subset ({KAGGLE_STYLEGAN_LIMIT} images)...')
kaggle_dir = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
print(f'Kaggle dir: {kaggle_dir}')

# Find the fake (StyleGAN2) images from Kaggle dataset
all_stylegan = sorted(
    glob.glob(f'{kaggle_dir}/**/fake/**/*.jpg', recursive=True) +
    glob.glob(f'{kaggle_dir}/**/fake/**/*.png', recursive=True)
)
print(f'StyleGAN2 images found: {len(all_stylegan)}')

# Sample to avoid imbalance
random.seed(42)
stylegan_sample = random.sample(all_stylegan, min(KAGGLE_STYLEGAN_LIMIT, len(all_stylegan)))
print(f'✅ Using {len(stylegan_sample)} StyleGAN2 fakes for mixing')

In [ ]:
# ── Cell 8: Build mixed dataset ───────────────────────────────────
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Combined data: (path, label)
# label: 1 = FAKE, 0 = REAL
all_data = (
    [(p, 1) for p in fake_paths] +          # inswapper face-swaps
    [(p, 1) for p in stylegan_sample] +     # StyleGAN2 fakes (balance)
    [(p, 0) for p in real_paths]            # Real faces
)

random.seed(42)
random.shuffle(all_data)

n_val   = int(len(all_data) * VAL_SPLIT)
val_data   = all_data[:n_val]
train_data = all_data[n_val:]

fake_count = sum(1 for _, l in train_data if l == 1)
real_count = sum(1 for _, l in train_data if l == 0)
print(f'Train: {len(train_data)} ({fake_count} fake, {real_count} real)')
print(f'Val:   {len(val_data)}')

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), torch.tensor(label, dtype=torch.float32)
        except Exception:
            # Skip corrupted images — return next valid one
            return self.__getitem__((idx + 1) % len(self.data))

train_loader = DataLoader(FaceDataset(train_data, train_tf), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FaceDataset(val_data,   val_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ DataLoaders ready — {len(train_loader)} train batches, {len(val_loader)} val batches')

In [ ]:
# ── Cell 9: Training setup ────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.BCEWithLogitsLoss()

# Start with backbone frozen — only train the head
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('✅ Optimizer ready')
print(f'   Backbone: FROZEN for first {FREEZE_EPOCHS} epochs')
print(f'   Head params: {sum(p.numel() for p in model.head.parameters())/1e3:.1f}K training')

In [ ]:
# ── Cell 10: Fine-tuning loop ─────────────────────────────────────
best_val_acc = 0.0
best_ckpt    = os.path.join(SAVE_DIR, 'efficientnet_b4_deepguard_v2.pth')
history      = []

for epoch in range(1, EPOCHS + 1):

    # Unfreeze backbone after FREEZE_EPOCHS
    if epoch == FREEZE_EPOCHS + 1:
        print(f'\n🔓 Epoch {epoch}: Unfreezing backbone...')
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = AdamW(model.parameters(), lr=LR * 0.3)  # Even lower LR for backbone
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS)

    # ── Train ──
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for step, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = (torch.sigmoid(logits) > 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)
        train_loss    += loss.item()

        if (step + 1) % 50 == 0:
            acc = train_correct / train_total * 100
            print(f'  Epoch {epoch} [{step+1}/{len(train_loader)}] loss={train_loss/(step+1):.4f} acc={acc:.2f}%')

    scheduler.step()
    train_acc = train_correct / train_total * 100

    # ── Validate ──
    model.eval()
    val_correct, val_total, val_loss_sum = 0, 0, 0.0
    tp, fp, fn, tn = 0, 0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            preds  = (torch.sigmoid(logits) > 0.5).float()

            val_correct  += (preds == labels).sum().item()
            val_total    += labels.size(0)
            val_loss_sum += loss.item()

            # Confusion matrix
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()

    val_acc = val_correct / val_total * 100
    precision = tp / (tp + fp + 1e-8) * 100
    recall    = tp / (tp + fn + 1e-8) * 100
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    history.append({'epoch': epoch, 'train_acc': train_acc, 'val_acc': val_acc, 'f1': f1})

    print(f'\n📊 Epoch {epoch}/{EPOCHS}')
    print(f'   Train acc: {train_acc:.2f}%')
    print(f'   Val   acc: {val_acc:.2f}%   F1: {f1:.2f}%   Prec: {precision:.2f}%   Recall: {recall:.2f}%')
    print(f'   Confusion: TP={tp} FP={fp} FN={fn} TN={tn}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f'   💾 New best saved → {best_ckpt}')

print(f'\n✅ Training complete! Best val acc: {best_val_acc:.2f}%')

In [ ]:
# ── Cell 11: Training summary ─────────────────────────────────────
print('\n📈 Training History:')
print(f'{"Epoch":<8}{"Train Acc":<14}{"Val Acc":<14}{"F1":<10}')
print('─' * 46)
for h in history:
    marker = ' ← best' if h['val_acc'] == best_val_acc else ''
    print(f"{h['epoch']:<8}{h['train_acc']:<14.2f}{h['val_acc']:<14.2f}{h['f1']:<10.2f}{marker}")

print(f'\nBest checkpoint: {best_ckpt}')
print(f'Best val acc:    {best_val_acc:.2f}%')

In [ ]:
# ── Cell 12: Test on individual images ────────────────────────────
# Quick sanity check before uploading
import torchvision.transforms as T

# Load best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
model.eval()

infer_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(image_path):
    img    = Image.open(image_path).convert('RGB')
    tensor = infer_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).item()
    label = 'FAKE' if prob > 0.5 else 'REAL'
    conf  = prob * 100 if label == 'FAKE' else (1 - prob) * 100
    return label, conf

# Test 5 random fake images
print('Testing on face-swap fakes:')
for p in random.sample(fake_paths, min(5, len(fake_paths))):
    label, conf = predict(p)
    status = '✅' if label == 'FAKE' else '❌'
    print(f'  {status} {os.path.basename(p)}: {label} ({conf:.1f}%)')

print('\nTesting on real images:')
for p in random.sample(real_paths, min(5, len(real_paths))):
    label, conf = predict(p)
    status = '✅' if label == 'REAL' else '❌'
    print(f'  {status} {os.path.basename(p)}: {label} ({conf:.1f}%)')

In [ ]:
# ── Cell 13: Upload v2 model to HuggingFace ───────────────────────
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

print('Uploading v2 model to HuggingFace...')
api.upload_file(
    path_or_fileobj=best_ckpt,
    path_in_repo='efficientnet_b4_deepguard_v2.pth',
    repo_id=HF_REPO,
    repo_type='model',
)

print(f'✅ v2 model live at: https://huggingface.co/{HF_REPO}')
print(f'   File: efficientnet_b4_deepguard_v2.pth')
print(f'   Best val acc: {best_val_acc:.2f}%')
print()
print('🎉 Done! Next step:')
print('   Download efficientnet_b4_deepguard_v2.pth from HuggingFace')
print('   Replace backend/models/efficientnet_b4_deepguard.pth')
print('   Update .env: DETECTION_MODEL_PATH=models/efficientnet_b4_deepguard_v2.pth')